In [5]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "font.size": 10})
SEED = 2026
print("NumPy:", np.__version__, "| Random seed:", SEED)

NumPy: 2.5.5 | Random seed: 2026


In [6]:
MODULATION = "BPSK"    # "BPSK" or "QPSK"
TRAIN_SYMBOLS = 5000      # known symbols for adaptive training
TEST_SYMBOLS = 12000      # unseen symbols for unbiased evaluation
SNR_DB = 18.0            # measured against channel output signal power
NUM_TAPS = 11            # number of LMS FIR weights
DECISION_DELAY = 4       # target is transmitted symbol s[n - D]
MU = 0.015               # LMS adaptation step size
CHANNEL = np.array([0.9, 0.50, -0.30], dtype=complex)
assert MODULATION in {"BPSK", "QPSK"}
assert 0 <= DECISION_DELAY < NUM_TAPS
assert TRAIN_SYMBOLS >= NUM_TAPS + DECISION_DELAY
assert MU > 0 and TEST_SYMBOLS > 0
print(f"{MODULATION}, {NUM_TAPS} taps, delay={DECISION_DELAY}, mu={MU}, SNR={SNR_DB} dB")

BPSK, 11 taps, delay=4, mu=0.015, SNR=18.0 dB


In [7]:
def generate_symbols(modulation, count, rng):
    if modulation == "BPSK":
        bits = rng.integers(0, 2, size=count, dtype=np.int8)
        symbols = (2 * bits - 1).astype(complex)
    elif modulation == "QPSK":
        bits = rng.integers(0, 2, size=(count, 2), dtype=np.int8)
        symbols = ((2 * bits[:, 0] - 1) + 1j * (2 * bits[:, 1] - 1)) / np.sqrt(2)
    else:
        raise ValueError("Modulation must be BPSK or QPSK")
    return bits, symbols

rng_demo = np.random.default_rng(SEED)
bits_demo, symbols_demo = generate_symbols(MODULATION, 12, rng_demo)
print("Transmitted bits:\n", bits_demo)
print("Mapped symbols:\n", symbols_demo)

Transmitted bits:
 [1 1 0 1 0 1 1 0 1 0 1 0]
Mapped symbols:
 [ 1.+0.j  1.+0.j -1.+0.j  1.+0.j -1.+0.j  1.+0.j  1.+0.j -1.+0.j  1.+0.j
 -1.+0.j  1.+0.j -1.+0.j]


In [8]:
def channel_fir(symbols, channel):
    return np.convolve(symbols, channel, mode="full")[:len(symbols)]

example_tx = np.array([1., -1., 1., 1., -1.], dtype=complex)
example_channel_out = channel_fir(example_tx, CHANNEL)
print("Channel h =", CHANNEL)
print("Example symbols =", example_tx)
print("Output with ISI =", example_channel_out)

Channel h = [ 0.9+0.j  0.5+0.j -0.3+0.j]
Example symbols = [ 1.+0.j -1.+0.j  1.+0.j  1.+0.j -1.+0.j]
Output with ISI = [ 0.9+0.j -0.4+0.j  0.1+0.j  1.7+0.j -0.7+0.j]


In [9]:
def add_awgn(clean_signal, snr_db, rng, real_noise=False):
    signal_power = np.mean(np.abs(clean_signal)**2)
    noise_power = signal_power / (10 ** (snr_db / 10))
    if real_noise:
        noise = np.sqrt(noise_power) * rng.standard_normal(len(clean_signal))
    else:
        noise = np.sqrt(noise_power / 2) * (
            rng.standard_normal(len(clean_signal)) + 1j*rng.standard_normal(len(clean_signal)))
    return clean_signal + noise

print("Example noisy samples:", np.round(add_awgn(example_channel_out, SNR_DB, np.random.default_rng(SEED)), 3))

Example noisy samples: [ 0.834-0.024j -0.38 -0.026j -0.058+0.025j  1.816-0.022j -0.647-0.019j]


In [10]:
def tapped_input(samples, n, num_taps):
    return samples[n - np.arange(num_taps)]

print("Example 3-tap window at index 3:", tapped_input(np.arange(10), 3, 3))

Example 3-tap window at index 3: [3 2 1]


In [11]:
def train_lms(received, transmitted, train_symbols, taps, delay, mu):
    if len(received) != len(transmitted):
        raise ValueError("Received and transmitted arrays must be the same length")
    if taps < 1 or delay < 0 or delay >= taps:
        raise ValueError("Use taps>=1 and 0 <= delay < taps")
    if train_symbols <= taps or train_symbols > len(transmitted):
        raise ValueError("Training interval is invalid")
    w = np.zeros(taps, dtype=complex)
    w[delay] = 1.0 + 0j  # initial through-path; not equalized yet
    errors, weights = [], []
    for n in range(max(taps - 1, delay), train_symbols + delay):
        if n >= len(received):
            break
        x = tapped_input(received, n, taps)
        desired = transmitted[n - delay]
        estimated = np.vdot(w, x)  # w^H x
        error = desired - estimated
        w += mu * x * np.conj(error)  # complex LMS
        if not np.all(np.isfinite(w)):
            raise FloatingPointError("LMS diverged: reduce MU")
        errors.append(abs(error) ** 2)
        weights.append(w.copy())
    return w, np.asarray(errors), np.asarray(weights)

print("LMS training function defined.")

LMS training function defined.
